# 🧪 Тестирование модуля макропрогнозирования

Этот ноутбук поэтапно тестирует модуль макропрогнозирования для компании **US Steel**.

## 0️⃣ Введение и инструкция

### Способ 1: Через Jupyter Notebook (рекомендуется)

1. **Откройте терминал** в корне проекта:
   ```bash
   cd /Users/arturhusnutdinov/Documents/IT\ Development/Docker/stressTest
   ```

2. **Запустите Jupyter Notebook**:
   ```bash
   jupyter notebook
   ```
   Это откроет браузер с Jupyter интерфейсом.

3. **Навигация в браузере**:
   - Перейдите в папку `companies/rusal/notebooks/`
   - Откройте файл `01_Test_Macro_Module.ipynb`

4. **Запуск ячеек**:
   - Нажмите `Shift + Enter` для выполнения ячейки
   - Или используйте меню: `Run` → `Run All Cells` для запуска всех ячеек
   - Или используйте кнопку `▶▶` для запуска всех ячеек

### Способ 2: Через JupyterLab

```bash
jupyter lab
```

Затем откройте ноутбук из файлового менеджера.

### Способ 3: Универсальный ноутбук (для другой компании)

Если вы используете универсальный ноутбук из `templates/notebooks/`:

1. Откройте `templates/notebooks/01_Test_Macro_Module.ipynb`
2. В разделе "Импорты и настройка" измените:
   ```python
   COMPANY = "rusal"  # Измените на имя вашей компании
   ```

## 🔍 Что тестируется

1. **Загрузка конфигурации** - проверка project.yaml и macro_ecm.yaml
2. **Проверка поиска файлов** - тест search_paths и file_map
3. **Чтение макро-факторов** - тест загрузки данных из CSV
4. **Анализ данных** - проверка временных рядов, пропусков, диапазонов
5. **Тест VECM модуля** - проверка инициализации и подготовки данных
6. **Настройки коинтеграции** - тест dummy переменных и стабильности прогнозов
7. **Запуск VECM** - полный pipeline макро-прогнозирования
8. **Проверка отчетов** - анализ коинтеграции и использованных методов
9. **Проверка выходных данных** - анализ созданных прогнозов
10. **Анализ методов** - статистика по VECM/ARIMA/RW методам
11. **Проверка трансформаций** - тест dln трансформаций

## ⚙️ Требования

- Python 3.7+
- Установленные зависимости: `pip install -r requirements.txt`
- Jupyter: `pip install jupyter` или `pip install jupyterlab`
- Компания должна быть настроена в `companies/{COMPANY}/configs/project.yaml`

## 💡 Советы

- Запускайте ячейки последовательно сверху вниз
- Если возникла ошибка, проверьте конфигурацию компании
- Результаты сохраняются в витрине данных (`FinancialDataMart`): таблицы `macro_forecasts`, `ecm_diagnostics`, `ecm_actual_vs_fitted`, `ecm_equations`.
- CSV-файлы используются только как legacy fallback и отображаются в соответствующих ячейках.

## 📊 Ожидаемые результаты

После успешного выполнения вы увидите:
- ✅ Все макро-файлы найдены и загружены
- ✅ Все данные валидированы (span, диапазоны)
- ✅ VECM прогнозы созданы для всех факторов
- ✅ Настройки коинтеграции проверены (dummy переменные, стабильность)
- ✅ Методы моделирования и переключения отражены в `ecm_diagnostics`
- ✅ Прогнозы и diagnostics доступны через витрину (ячейки ниже их отображают)
- ✅ Legacy отчёт `outputs/logs/vecm_run_report.md` доступен для анализа




## 📋 Оглавление

- [1️⃣ Подготовка окружения](#section-1)
- [2️⃣ Загрузка конфигураций](#section-2)
- [3️⃣ Проверка источников данных](#section-3)
- [4️⃣ История и анализ макро-факторов](#section-4)
- [5️⃣ Конфигурация VECM](#section-5)
- [6️⃣ Подготовка данных для VECM](#section-6)
- [7️⃣ Запуск VECM pipeline](#section-7)
- [8️⃣ Анализ результатов VECM](#section-8)
- [9️⃣ Отчёты и dummy](#section-9)
- [🔟 Итоговая сводка](#section-10)
- [1️⃣1️⃣ Итоговый отчет](#section-11)


## 1️⃣ Подготовка окружения <a id="section-1"></a>


In [ ]:
import sys
from pathlib import Path
from typing import Optional
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import HTML, Markdown, display

# Настройка для графиков в Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

# Определение корня проекта
# Ноутбук находится в: companies/rusal/notebooks/
# Отсюда нужно попасть в корень проекта где находится engine/

current_dir = Path.cwd()
# Проверяем где мы запущены
if (current_dir / "engine").exists():
    # Если запущены из корня проекта
    ROOT = current_dir
elif (current_dir.parent / "engine").exists():
    # Если на уровень выше корня
    ROOT = current_dir.parent
elif (current_dir / "companies" / "rusal" / "notebooks").exists():
    # Если мы в папке companies/rusal/notebooks/
    ROOT = current_dir.parent.parent.parent  # companies/rusal/notebooks/ -> ../../
elif (current_dir / "notebooks").exists():
    # Если мы в папке notebooks/
    ROOT = current_dir.parent.parent.parent  # notebooks/ -> rusal/ -> companies/ -> ../
else:
    # Fallback: предполагаем что ноутбук в companies/rusal/notebooks/
    # и нужно подняться на 3 уровня
    ROOT = Path('../../..').resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from engine.database.data_mart import get_data_mart

# Определяем COMPANY автоматически из пути ноутбука
# Пытаемся найти companies/ в пути
COMPANY = "rusal"  # fallback по умолчанию
if "companies" in current_dir.parts:
    companies_idx = current_dir.parts.index("companies")
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / "companies" / COMPANY

print(f"📁 ROOT: {ROOT}")
print(f"📁 ROOT существует: {ROOT.exists()}")
print(f"📁 engine/ существует: {(ROOT / 'engine').exists()}")
print(f"🏢 COMPANY: {COMPANY}")
print(f"📁 Company root: {croot}")
print(f"📁 Company root существует: {croot.exists()}")
print(f"📄 project.yaml: {(croot / 'configs' / 'project.yaml').exists()}")

# Проверка что все правильно настроено
if not (ROOT / 'engine').exists():
    print(f"\n⚠️ ВНИМАНИЕ: engine/ не найден в {ROOT}")
    print(f"   Текущая директория: {Path.cwd()}")
    print(f"   Попробуйте запустить ноутбук из корня проекта")
    print(f"   или укажите правильный ROOT вручную")


## 2️⃣ Загрузка конфигураций <a id="section-2"></a>


In [ ]:
# Загрузка project.yaml
proj_path = croot / "configs" / "project.yaml"

# Проверяем существование файла
if not proj_path.exists():
    print(f"❌ ОШИБКА: project.yaml не найден: {proj_path}")
    print(f"   ROOT: {ROOT}")
    print(f"   COMPANY: {COMPANY}")
    print(f"   croot: {croot}")
    print(f"   Проверьте путь к проекту!")
    proj = {}
else:
    try:
        with open(proj_path, 'r', encoding='utf-8') as f:
            proj = yaml.safe_load(f) or {}
        if not proj:
            print(f"⚠️ ПРЕДУПРЕЖДЕНИЕ: project.yaml пуст или не парсится: {proj_path}")
    except Exception as e:
        print(f"❌ ОШИБКА при чтении project.yaml: {e}")
        proj = {}

macro_config = proj.get("macro_forecast", {})
factors = macro_config.get("factors", [])
file_map = macro_config.get("file_map", {})
search_paths = macro_config.get("search_paths", [])
macro_ecm_config_path = macro_config.get("config", "configs/forecast/macro_ecm.yaml")

# Разрешаем путь к конфигу (может быть относительным или абсолютным)
if macro_ecm_config_path.startswith("companies/"):
    ecm_config_path = ROOT / macro_ecm_config_path
elif macro_ecm_config_path.startswith("configs/"):
    ecm_config_path = ROOT / macro_ecm_config_path
else:
    ecm_config_path = croot / macro_ecm_config_path

display(Markdown("### Конфигурация из project.yaml"))

# Детальная диагностика
if not proj:
    print(f"❌ Конфигурация не загружена!")
elif not macro_config:
    print(f"❌ Секция 'macro_forecast' отсутствует в project.yaml")
    print(f"   Доступные секции: {list(proj.keys())}")
else:
    print(f"✅ Файл загружен: {proj_path.relative_to(ROOT)}")
    print(f"✅ Факторы ({len(factors)}): {factors}")
    print(f"\nFile Map ({len(file_map)} записей):")
    for k, v in file_map.items():
        print(f"  - {k}: {v}")
    print(f"\nSearch Paths ({len(search_paths)} путей):")
    for sp in search_paths:
        print(f"  - {sp}")
    print(f"\nConfig path: {macro_ecm_config_path}")
    print(f"Resolved config path: {ecm_config_path.relative_to(ROOT) if ecm_config_path.exists() else 'НЕ НАЙДЕН'}")
    if not ecm_config_path.exists():
        print(f"   Полный путь: {ecm_config_path}")


## 3️⃣ Проверка источников данных <a id="section-3"></a>


In [ ]:
from engine.ecm.core import _find_driver_csv

macro_history: dict[str, dict[int, float]] = {}
macro_forecasts: dict[str, dict[int, float]] = {}

history_rows = []
forecast_rows = []

with get_data_mart(ROOT, COMPANY) as mart:
    for factor in factors:
        meta = mart.get_macro_factor_meta(factor)
        default_scope = meta.default_scope if meta else "global"
        industry_code = meta.industry_code if meta and meta.industry_code else mart.company_industry

        history = mart.get_macro_factor(factor, scope=default_scope, industry=industry_code)
        scope_used = default_scope
        if not history and default_scope != "global":
            history = mart.get_macro_factor(factor, scope="global")
            if history:
                scope_used = "global"
        if not history and default_scope == "company":
            history = mart.get_macro_factor(factor, scope="company")
            if history:
                scope_used = "company"

        history_years: list[int] = sorted(history.keys()) if history else []
        history_rows.append({
            "factor": factor,
            "default_scope": default_scope,
            "scope_used": scope_used if history else "missing",
            "industry": industry_code or "",
            "meta_source": meta.source if meta and meta.source else "",
            "years_count": len(history_years),
            "start_year": history_years[0] if history_years else None,
            "end_year": history_years[-1] if history_years else None,
        })

        if history:
            macro_history[factor] = history

        scenario = "baseline"
        forecast_industry = industry_code or ""
        forecast = mart.get_macro_forecast(factor, scenario=scenario, industry=industry_code)
        if not forecast and forecast_industry:
            forecast = mart.get_macro_forecast(factor, scenario=scenario)
            if forecast:
                forecast_industry = ""

        forecast_years: list[int] = sorted(forecast.keys()) if forecast else []
        forecast_rows.append({
            "factor": factor,
            "scenario": scenario,
            "industry": forecast_industry,
            "horizon_years": len(forecast_years),
            "start_year": forecast_years[0] if forecast_years else None,
            "end_year": forecast_years[-1] if forecast_years else None,
        })

        if forecast:
            macro_forecasts[factor] = forecast

display(Markdown("### 📦 Источники макроданных (Data Mart)"))
display(pd.DataFrame(history_rows))

display(Markdown("### 📈 Макропрогнозы в витрине"))
display(pd.DataFrame(forecast_rows))

# Legacy fallback: проверяем наличие CSV только для диагностики
legacy_results = []
for factor in factors:
    file_path = _find_driver_csv(ROOT, COMPANY, factor)
    if file_path and file_path.exists():
        legacy_results.append({
            'factor': factor,
            'csv_path': str(file_path.relative_to(ROOT)),
            'size': f"{file_path.stat().st_size} bytes",
            'note': 'legacy'
        })

if legacy_results:
    display(Markdown("### ♻️ CSV (legacy fallback)"))
    display(pd.DataFrame(legacy_results))
    print("ℹ️ Рекомендуется перенести данные в витрину и отказаться от CSV.")
else:
    print("✅ CSV-файлы не используются (все факторы берутся из витрины).")


## 4️⃣ История и анализ макро-факторов <a id="section-4"></a>


### 4.1 Загрузка макроданных

In [ ]:
history_summary_lines: list[str] = []
history_table_rows: list[dict[str, object]] = []

macro_history_local = macro_history if 'macro_history' in globals() else {}
macro_forecasts_local = macro_forecasts if 'macro_forecasts' in globals() else {}

for factor in factors:
    history = macro_history_local.get(factor, {}) or {}
    forecast = macro_forecasts_local.get(factor, {}) or {}

    history_years = sorted(int(year) for year in history.keys() if history.get(year) is not None)
    years_count = len(history_years)
    start_year = history_years[0] if history_years else None
    end_year = history_years[-1] if history_years else None

    forecast_years = sorted(int(year) for year in forecast.keys() if forecast.get(year) is not None)
    last_year_candidates: list[int] = []
    if end_year is not None:
        last_year_candidates.append(end_year)
    if forecast_years:
        last_year_candidates.append(forecast_years[-1])
    last_year = max(last_year_candidates) if last_year_candidates else None

    span_years = (end_year - start_year + 1) if start_year is not None and end_year is not None else None

    values = [float(history.get(year)) for year in history_years if history.get(year) is not None]
    min_value = min(values) if values else None
    max_value = max(values) if values else None

    missing_count = 0
    if start_year is not None and end_year is not None:
        expected_years = set(range(start_year, end_year + 1))
        missing_count = len(expected_years - set(history_years))

    status = "✅" if years_count else "⚠️"
    data_source = "Data Mart (history)" if years_count else "Not found"

    if years_count and start_year is not None and end_year is not None:
        history_summary_lines.append(
            f"{status} {factor}: {years_count} лет данных ({start_year}-{end_year}) [{data_source}]"
        )
    else:
        history_summary_lines.append(f"{status} {factor}: история отсутствует [{data_source}]")

    history_table_rows.append({
        "factor": factor,
        "status": status,
        "data_source": data_source,
        "years_count": years_count,
        "start_year": start_year,
        "end_year": end_year,
        "last_year": last_year,
        "span_years": span_years,
        "min_value": min_value,
        "max_value": max_value,
        "missing_count": missing_count,
    })

for line in history_summary_lines:
    print(line)

history_df = pd.DataFrame(history_table_rows)
if not history_df.empty:
    display(history_df[[
        "factor",
        "status",
        "data_source",
        "years_count",
        "start_year",
        "end_year",
        "last_year",
        "span_years",
        "min_value",
        "max_value",
        "missing_count",
    ]])
else:
    print("⚠️ Исторические данные не найдены ни для одного фактора.")

# Обновляем глобальный словарь макроданных для последующих тестов
macro_data = {factor: macro_history_local.get(factor, {}) or {} for factor in factors}


### 4.2 Анализ исторических данных

#### 4.2.1 Графики приращений ln

In [ ]:
rows = max(1, (len(factors) + 1) // 2)
cols = 2 if len(factors) > 1 else 1
fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows), constrained_layout=True)
if isinstance(axes, np.ndarray):
    axes_list = axes.flatten().tolist()
elif isinstance(axes, (list, tuple)):
    axes_list = list(axes)
else:
    axes_list = [axes]

for idx, factor in enumerate(factors):
    if idx >= len(axes_list):
        break
    ax = axes_list[idx]

    history = macro_history.get(factor, {}) if 'macro_history' in globals() else {}
    series = pd.Series(history).sort_index() if history else pd.Series(dtype=float)

    if series.empty:
        ax.set_title(f"{factor} — нет данных")
        ax.axis('off')
        continue

    positive = series[series > 0].dropna()
    if positive.empty:
        ax.set_title(f"{factor} — неположительные значения")
        ax.axis('off')
        continue

    ln_series = np.log(positive)
    dln_series = ln_series.diff().dropna()

    if dln_series.empty:
        ax.set_title(f"{factor} — недостаточно точек для Δln")
        ax.axhline(0, color='gray', linewidth=0.8)
        continue

    ax.plot(dln_series.index, dln_series.values, marker='o', linewidth=1.2, color='#2563eb')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(f"{factor}")
    ax.set_xlabel('Год')
    ax.set_ylabel('Δln')

for j in range(len(factors), len(axes_list)):
    axes_list[j].axis('off')

plt.show()


##### 4.2.2 Таблица данных (первые и последние годы)

In [ ]:
history_first_last_rows: list[dict[str, object]] = []

for factor in factors:
    history = macro_history.get(factor, {}) if 'macro_history' in globals() else {}
    if not history:
        continue

    series = pd.Series(history).sort_index().dropna()
    if series.empty:
        continue

    first_year = int(series.index.min())
    last_year = int(series.index.max())

    first_value = float(series.iloc[0])
    last_value = float(series.iloc[-1])

    second_year = int(series.index[1]) if len(series) > 1 else None
    second_value = float(series.iloc[1]) if len(series) > 1 else None

    second_last_year = int(series.index[-2]) if len(series) > 1 else None
    second_last_value = float(series.iloc[-2]) if len(series) > 1 else None

    history_first_last_rows.append({
        'factor': factor,
        'first_year': first_year,
        'first_value': first_value,
        'second_year': second_year,
        'second_value': second_value,
        'second_last_year': second_last_year,
        'second_last_value': second_last_value,
        'last_year': last_year,
        'last_value': last_value,
    })

if history_first_last_rows:
    history_first_last_df = pd.DataFrame(history_first_last_rows)
    display(history_first_last_df)
else:
    print('⚠️ Недостаточно данных для отображения таблицы первых и последних значений.')


##### 4.2.3 Статистика

In [ ]:
if history_first_last_rows:
    stats_rows: list[dict[str, object]] = []

    for factor in factors:
        history = macro_history.get(factor, {}) if 'macro_history' in globals() else {}
        if not history:
            continue

        series = pd.Series(history).sort_index().dropna()
        if series.empty:
            continue

        stats_rows.append({
            'factor': factor,
            'count': int(series.count()),
            'mean': float(series.mean()),
            'std': float(series.std(ddof=0)),
            'min': float(series.min()),
            '25%': float(series.quantile(0.25)),
            '50%': float(series.quantile(0.50)),
            '75%': float(series.quantile(0.75)),
            'max': float(series.max()),
        })

    if stats_rows:
        stats_df = pd.DataFrame(stats_rows)
        display(stats_df)
    else:
        print('⚠️ Недостаточно данных для расчёта статистики.')
else:
    print('⚠️ Сначала сформируйте таблицу 4.2.2, чтобы рассчитать статистику.')


## 5️⃣ Конфигурация VECM <a id="section-5"></a>



In [ ]:
# Проверяем что можно импортировать VECM функции
try:
    from engine.ecm.vecm import run_vecm_all, _stack_block_ln
    from engine.ecm.vecm import _to_ln_series
    print("✅ VECM модуль импортирован успешно")
except Exception as e:
    print(f"❌ Ошибка импорта VECM: {e}")
    import traceback
    traceback.print_exc()

# Читаем конфигурацию VECM
ecm_config_path = ROOT / macro_ecm_config_path
if not ecm_config_path.exists():
    ecm_config_path = ROOT / "configs" / "forecast" / "macro_ecm.yaml"

display(Markdown("### Конфигурация VECM"))
if ecm_config_path.exists():
    ecm_cfg = yaml.safe_load(ecm_config_path.read_text(encoding='utf-8'))
    print(f"Config path: {ecm_config_path.relative_to(ROOT)}")
    print(f"\nНастройки:")
    print(f"  - use_log: {ecm_cfg.get('run', {}).get('use_log', 'N/A')}")
    print(f"  - short_run_transform: {ecm_cfg.get('run', {}).get('short_run_transform', 'N/A')}")
    print(f"  - require_span_years: {ecm_cfg.get('run', {}).get('require_span_years', 'N/A')}")
    print(f"  - start_year_required: {ecm_cfg.get('run', {}).get('start_year_required', 'N/A')}")
    print(f"  - resid_ar1: {ecm_cfg.get('run', {}).get('resid_ar1', True)}")
    print(f"  - resid_arma: {ecm_cfg.get('run', {}).get('resid_arma', False)}")
    
    vecm_config = ecm_cfg.get('vecm', {})
    blocks = vecm_config.get('groups', [])
    
    if blocks:
        print(f"\nVECM Groups ({len(blocks)} групп):")
        # groups может быть списком словарей или списком списков
        for i, block in enumerate(blocks):
            if isinstance(block, dict):
                block_name = block.get('name', f'group_{i}')
                block_factors = block.get('factors', [])
            elif isinstance(block, list):
                block_name = f'group_{i}'
                block_factors = block
            else:
                block_name = f'group_{i}'
                block_factors = []
            print(f"  - {block_name}: {block_factors}")
    else:
        # Проверяем старый формат (blocks как словарь)
        blocks_old = ecm_cfg.get('blocks', {})
        if blocks_old:
            print(f"\nVECM Blocks (старый формат, {len(blocks_old)} блоков):")
            for block_name, block_data in blocks_old.items():
                if isinstance(block_data, dict):
                    block_factors = block_data.get('factors', [])
                else:
                    block_factors = block_data if isinstance(block_data, list) else []
                print(f"  - {block_name}: {block_factors}")
        else:
            print(f"  ⚠️ VECM groups/blocks не найдены в конфиге")

    auto_select_cfg = vecm_config.get('auto_select', {})
    if auto_select_cfg:
        print(f"\n🤖 Автоподбор групп:")
        print(f"  - enabled: {auto_select_cfg.get('enabled', False)}")
        print(f"  - min_group_size: {auto_select_cfg.get('min_group_size', 'N/A')}")
        print(f"  - max_group_size: {auto_select_cfg.get('max_group_size', 'N/A')}")
        print(f"  - max_groups: {auto_select_cfg.get('max_groups', 'N/A')}")
        print(f"  - max_combinations: {auto_select_cfg.get('max_combinations', 'N/A')}")
        print(f"  - prefer_manual: {auto_select_cfg.get('prefer_manual', True)}")
        print(f"  - score_metric: {auto_select_cfg.get('score_metric', 'aic')}")
    
    # Проверяем настройки lag_search
    lag_search_cfg = ecm_cfg.get('lag_search', {})
    if lag_search_cfg:
        print(f"\n🔍 Настройки lag_search:")
        print(f"  - p_min: {lag_search_cfg.get('p_min', 'N/A')}")
        print(f"  - p_max: {lag_search_cfg.get('p_max', 'N/A')}")
    
    # Проверяем настройки rank_test
    rank_test_cfg = ecm_cfg.get('rank_test', {})
    if rank_test_cfg:
        print(f"\n📏 Настройки rank_test:")
        print(f"  - method: {rank_test_cfg.get('method', 'N/A')}")
        print(f"  - alpha: {rank_test_cfg.get('alpha', 'N/A')}")
    
    # Проверяем настройки diagnostics
    diagnostics_cfg = ecm_cfg.get('diagnostics', {})
    if diagnostics_cfg:
        print(f"\n🔬 Настройки diagnostics:")
        print(f"  - ljung_box_alpha: {diagnostics_cfg.get('ljung_box_alpha', 'N/A')}")
    
    # Проверяем настройки коинтеграционного тестирования
    coint_test_cfg = ecm_cfg.get('cointegration_testing', {})
    if coint_test_cfg:
        print(f"\n📊 Настройки тестирования коинтеграции:")
        print(f"  - use_dummies: {coint_test_cfg.get('use_dummies', False)}")
        print(f"  - shock_years: {coint_test_cfg.get('shock_years', [])}")
        print(f"  - max_dummies: {coint_test_cfg.get('max_dummies', 0)}")
        print(f"  - dummy_aic_penalty: {coint_test_cfg.get('dummy_aic_penalty', 0)}")
        print(f"  - check_forecast_stability: {coint_test_cfg.get('check_forecast_stability', False)}")
        print(f"  - stability_max_change_pct: {coint_test_cfg.get('stability_max_change_pct', 0)}")
        print(f"  - stability_max_volatility: {coint_test_cfg.get('stability_max_volatility', 0)}")
        print(f"  - stability_max_trend_slope: {coint_test_cfg.get('stability_max_trend_slope', 0)}")
        print(f"  - switch_to_univariate_if_unstable: {coint_test_cfg.get('switch_to_univariate_if_unstable', False)}")

    fallback_cfg = ecm_cfg.get('fallback', {})
    if fallback_cfg:
        print(f"\n🛟 Настройки fallback для унивариантных моделей:")
        print(f"  - order: {fallback_cfg.get('order', [])}")
        auto_arima_cfg = fallback_cfg.get('auto_arima', {})
        if auto_arima_cfg:
            print(f"    • auto_arima: max_p={auto_arima_cfg.get('max_p', 'N/A')}, max_q={auto_arima_cfg.get('max_q', 'N/A')}, max_d={auto_arima_cfg.get('max_d', 'N/A')}, seasonal={auto_arima_cfg.get('seasonal', False)}")
        ets_cfg = fallback_cfg.get('ets', {})
        if ets_cfg:
            print(f"    • ets: trend={ets_cfg.get('trend', 'add')}, damped_trend={ets_cfg.get('damped_trend', False)}, seasonal={ets_cfg.get('seasonal', None)}")
        linear_cfg = fallback_cfg.get('linear_reg', {})
        if linear_cfg:
            print(f"    • linear_reg: window={linear_cfg.get('window', 'N/A')} (enabled={linear_cfg.get('enabled', True)})")
else:
    print(f"❌ Конфиг не найден: {ecm_config_path}")



## 6️⃣ Подготовка данных для VECM <a id="section-6"></a>



In [ ]:
# Тестируем преобразование в ln
display(Markdown("### Преобразование уровней в ln"))

try:
    from engine.ecm.vecm import _to_ln_series
    
    ln_test_results = []
    for factor, data in list(macro_data.items())[:3]:  # Тестируем первые 3 фактора
        if data:
            ln_series = _to_ln_series(data)
            if ln_series is not None and len(ln_series) > 0:
                ln_test_results.append({
                    'factor': factor,
                    'status': '✅',
                    'ln_years': len(ln_series),
                    'ln_min': float(ln_series.min()),
                    'ln_max': float(ln_series.max()),
                    'has_negative': any(v < 0 for v in data.values() if pd.notna(v) and v > 0)
                })
                print(f"✅ {factor}: {len(ln_series)} лет в ln ({ln_series.index.min()}-{ln_series.index.max()})")
            else:
                ln_test_results.append({'factor': factor, 'status': '❌', 'error': 'Empty ln_series'})
                print(f"❌ {factor}: Ошибка преобразования в ln")
    
    ln_df = pd.DataFrame(ln_test_results)
    if not ln_df.empty:
        display(ln_df)
except Exception as e:
    print(f"❌ Ошибка тестирования ln-трансформации: {e}")
    import traceback
    traceback.print_exc()



## 7️⃣ Запуск VECM pipeline <a id="section-7"></a>

Перед запуском очищаем предыдущие прогнозы и формируем полный прогнозный конвейер.

### 7.1 Запуск полного VECM pipeline

In [ ]:
scenario = macro_config.get("scenario", "baseline") if isinstance(macro_config, dict) else "baseline"
scenario = scenario or "baseline"

with get_data_mart(ROOT, COMPANY) as mart:
    cursor = mart.db.conn.cursor()
    cursor.execute("DELETE FROM macro_forecasts WHERE company = ?", (COMPANY,))
    mart.db.conn.commit()
    print(f"🧹 Очищены прогнозы macro_forecasts для {COMPANY}")

try:
    from engine.ecm.vecm import run_vecm_all

    print(f"Запуск VECM для {COMPANY}...")
    print(f"Config: {ecm_config_path}")

    summary_df = run_vecm_all(
        root=str(ROOT),
        company=COMPANY,
        cfg_path=str(ecm_config_path)
    )

    print("✅ VECM pipeline выполнен успешно!")

    with get_data_mart(ROOT, COMPANY) as mart:
        diag_df_latest = mart.get_ecm_diagnostics()
        forecast_counts = {
            factor: len(mart.get_macro_forecast(factor))
            for factor in factors
        }

    print("\n📦 Обновления витрины:")
    print(f"  - diagnostics записей: {len(diag_df_latest)}")
    print(f"  - факторов с прогнозом: {sum(1 for v in forecast_counts.values() if v > 0)}")
except Exception as e:
    print(f"❌ Ошибка запуска VECM: {e}")
    import traceback
    traceback.print_exc()


## 8️⃣ Анализ результатов VECM <a id="section-8"></a>

Ниже загружаем результаты из витрины и анализируем макропрогноз.

### 8.1 Анализ макропрогноза (витрина данных)

In [ ]:
selection_log_path = ROOT / "companies" / COMPANY / "outputs" / "logs" / "vecm_group_selection.json"
if selection_log_path.exists():
    try:
        selection_data = json.loads(selection_log_path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать vecm_group_selection.json: {exc}")
    else:
        display(Markdown("### 🤖 Лог автоподбора групп"))
        if selection_data:
            selection_df = pd.DataFrame(selection_data)
            if 'factors' in selection_df.columns:
                selection_df['factors'] = selection_df['factors'].apply(
                    lambda v: ', '.join(v) if isinstance(v, list) else v
                )
            display(selection_df)
        else:
            print("ℹ️ Лог автоподбора пуст.")
else:
    print("ℹ️ Файл vecm_group_selection.json еще не создан — запустите VECM для генерации лога.")



### 8.2 Анализ точности прогнозов

In [ ]:
ecm_diag_df: pd.DataFrame = pd.DataFrame()
ecm_forecast_diag_df: pd.DataFrame = pd.DataFrame()
ecm_avf_df: pd.DataFrame = pd.DataFrame()
ecm_equations_df: pd.DataFrame = pd.DataFrame()
macro_forecasts: dict[str, dict[int, float]] = {}

with get_data_mart(ROOT, COMPANY) as mart:
    diag_raw = mart.get_ecm_diagnostics()
    ecm_diag_df = diag_raw if isinstance(diag_raw, pd.DataFrame) else pd.DataFrame()

    forecast_diag_raw = mart.get_ecm_forecast_diag()
    ecm_forecast_diag_df = forecast_diag_raw if isinstance(forecast_diag_raw, pd.DataFrame) else pd.DataFrame()

    avf_raw = mart.get_ecm_actual_vs_fitted()
    ecm_avf_df = avf_raw if isinstance(avf_raw, pd.DataFrame) else pd.DataFrame()

    equations = mart.get_ecm_equations()
    if isinstance(equations, pd.DataFrame) and not equations.empty:
        ecm_equations_df = equations
    else:
        ecm_equations_df = pd.DataFrame()

    macro_forecasts = {
        factor: mart.get_macro_forecast(factor) or {}
        for factor in factors
    }

forecast_summary = []
example_factor: str | None = None
example_diag_row: pd.DataFrame | None = None

for factor in factors:
    diag_row = ecm_diag_df[ecm_diag_df['factor_name'] == factor].tail(1) if not ecm_diag_df.empty else pd.DataFrame()
    span_end = int(diag_row['span_end'].iloc[0]) if not diag_row.empty and pd.notna(diag_row['span_end'].iloc[0]) else None
    method = diag_row['method'].iloc[0] if not diag_row.empty else 'N/A'
    note = diag_row['note'].iloc[0] if not diag_row.empty else ''

    forecast_series = macro_forecasts.get(factor, {})
    years = sorted(forecast_series.keys())
    forecast_horizon = len([y for y in years if span_end is not None and y > span_end])
    forecast_end = max(years) if years else None
    has_forecast = forecast_horizon > 0

    forecast_summary.append({
        'factor': factor,
        'method': method,
        'span_end': span_end,
        'forecast_years': forecast_horizon,
        'forecast_end': forecast_end,
        'note': note
    })

    if example_factor is None and has_forecast:
        example_factor = factor
        example_diag_row = diag_row

summary_df = pd.DataFrame(forecast_summary)
if not summary_df.empty:
    display(Markdown("#### 8.1.1 Сводка прогнозов"))
    display(summary_df)
else:
    print("⚠️ В витрине нет данных о макропрогнозе.")

example_series = pd.Series(dtype=float)
example_span_end: int | None = None
if example_factor:
    display(Markdown(f"#### 8.1.2 Пример прогноза ({example_factor})"))

    example_series = pd.Series(macro_forecasts.get(example_factor, {})).sort_index()
    if example_diag_row is not None and not example_diag_row.empty:
        span_val = example_diag_row['span_end'].iloc[0]
        example_span_end = int(span_val) if pd.notna(span_val) else None

    if not example_series.empty:
        df_display = pd.DataFrame({'year': example_series.index, 'ln_value': example_series.values})
        display(df_display.tail(10))

        try:
            import matplotlib.pyplot as plt

            history_mask = example_series.index <= example_span_end if example_span_end is not None else example_series.index <= example_series.index.max()
            history_series = example_series[history_mask]
            forecast_series = example_series[~history_mask] if example_span_end is not None else pd.Series(dtype=float)

            plt.figure(figsize=(12, 6))
            if not history_series.empty:
                plt.plot(history_series.index, history_series.values, 'o-', label='История (ln)', linewidth=2, markersize=6)
            if not forecast_series.empty:
                plt.plot(forecast_series.index, forecast_series.values, 's--', label='Прогноз (ln)', linewidth=2, markersize=6, color='green')
            plt.xlabel('Год')
            plt.ylabel('ln(значение)')
            plt.title(f'Прогноз для {example_factor}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"⚠️ Не удалось построить график прогноза: {exc}")
    else:
        print(f"⚠️ Для {example_factor} прогнозные значения отсутствуют.")
else:
    print("⚠️ Не найден фактор с прогнозным горизонтом для примера.")

display(Markdown("### 8.2 Анализ точности прогнозов"))

if example_factor:
    avf_rows = ecm_avf_df[ecm_avf_df['factor_name'] == example_factor] if not ecm_avf_df.empty else pd.DataFrame()
    if not avf_rows.empty:
        display(Markdown(f"#### Actual vs Fitted для {example_factor}"))
        display(avf_rows.head(10))

        try:
            import matplotlib.pyplot as plt

            plt.figure(figsize=(12, 6))
            plt.plot(avf_rows['year'], avf_rows['actual_ln'], 'o-', label='Actual (ln)', linewidth=2, markersize=6)
            plt.plot(avf_rows['year'], avf_rows['fitted_ln'], 's--', label='Fitted (ln)', linewidth=2, markersize=6, color='orange')
            plt.xlabel('Год')
            plt.ylabel('ln(значение)')
            plt.title(f'Actual vs Fitted для {example_factor}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"⚠️ Не удалось построить график Actual vs Fitted: {exc}")
    else:
        print(f"ℹ️ Для {example_factor} нет записей actual vs fitted в витрине.")
else:
    print("ℹ️ Нет примера фактора с горизонтом прогноза — анализ точности пропущен.")


## 9️⃣ Отчёты и dummy <a id="section-9"></a>

Собираем отчёты по VECM, рекомендации по dummy и анализ аномалий.

### 9.1 📊 Анализ VECM Run Report

In [ ]:
# Проверяем VECM отчет для анализа коинтеграции и dummy
log_dir = croot / "outputs" / "logs"
vecm_report_path = log_dir / "vecm_run_report.md"

if vecm_report_path.exists():
    report_text = vecm_report_path.read_text(encoding='utf-8')
    lines = report_text.split('\n')

    display(Markdown("#### Отчет VECM:"))
    print("=" * 80)
    for line in lines:
        if line.strip():
            print(line)
    print("=" * 80)

    if 'dummies' in report_text.lower():
        display(Markdown("\n#### 🎯 Анализ dummy переменных:"))
        dummies_lines = [l for l in lines if 'dummy' in l.lower() or 'shock' in l.lower()]
        if dummies_lines:
            for line in dummies_lines:
                if line.strip():
                    print(f"  {line}")
        else:
            print("  Dummy переменные не использовались (базовая VECM дала лучший AIC)")

    if any(keyword in report_text.lower() for keyword in ['unstable', 'stable', 'switching']):
        display(Markdown("\n#### 📈 Анализ стабильности прогнозов:"))
        stability_lines = [l for l in lines if any(k in l.lower() for k in ['stable', 'unstable', 'switching'])]
        if stability_lines:
            for line in stability_lines:
                if line.strip():
                    print(f"  {line}")
        else:
            print("  Все прогнозы стабильны, переход на univariate ECM не потребовался")

    if 'ecm_equations_df' in globals() and isinstance(ecm_equations_df, pd.DataFrame) and not ecm_equations_df.empty:
        display(Markdown("\n#### 📐 Уравнения коинтеграции (витрина данных):"))
        display(ecm_equations_df)
    else:
        print("\nℹ️ Уравнения коинтеграции в витрине отсутствуют или еще не сохранены.")
else:
    display(Markdown("#### ⚠️ Отчет VECM не найден"))
    print(f"Ожидалось: {vecm_report_path}")
    print("Запустите VECM pipeline для создания отчета")


### 9.2 🔄 Рекомендации по dummy

In [ ]:
anomaly_df = pd.DataFrame()
metrics_df = pd.DataFrame()

try:
    with get_data_mart(ROOT, COMPANY) as mart:
        anomaly_df = mart.get_macro_anomalies()
        metrics_df = mart.get_macro_anomaly_metrics()
except Exception:
    anomaly_df = pd.DataFrame()
    metrics_df = pd.DataFrame()

if not anomaly_df.empty:
    display(Markdown("Данные из витрины macro_anomalies"))
    display(anomaly_df)
    if not metrics_df.empty:
        display(Markdown("Сводные метрики по факторам"))
        display(metrics_df)
    suggested_years = sorted({
        int(row["year"])
        for _, row in anomaly_df.iterrows()
        if int(row.get("suggested_dummy", 0)) == 1
    })
    if suggested_years:
        print(f"Рекомендованные годы к добавлению: {suggested_years}")
    else:
        print("Автодетект не нашёл кандидатов для новых dummy.")
else:
    if macro_df.empty:
        print("⚠️ Исторические ряды отсутствуют; рекомендации по dummy невозможно сформировать.")
    else:
        coint_test_cfg = ecm_cfg.get('cointegration_testing', {}) if 'ecm_cfg' in globals() else {}
        current_shocks = sorted({int(y) for y in coint_test_cfg.get('shock_years', []) if y is not None})

        macro_df_sorted = macro_df.sort_index()
        yoy_changes = macro_df_sorted.diff()
        abs_changes = yoy_changes.abs()

        if abs_changes.dropna(how='all').empty:
            print("⚠️ Недостаточно данных для оценки шоков.")
        else:
            baseline_quantile = abs_changes.stack().quantile(0.90) if not abs_changes.stack().empty else np.nan
            threshold = float(max(0.08, baseline_quantile)) if not np.isnan(baseline_quantile) else 0.08

            candidate_rows: list[dict[str, object]] = []
            for year in abs_changes.index[1:]:
                row = abs_changes.loc[year].dropna()
                strong = row[row >= threshold]
                if strong.empty:
                    continue

                candidate_rows.append({
                    "year": int(year),
                    "count": int(len(strong)),
                    "avg_abs_change": float(strong.mean()),
                    "max_abs_change": float(strong.max()),
                    "factors": ", ".join(strong.index.tolist())
                })

            if not candidate_rows:
                print(f"⚠️ Не найдено лет с отклонением выше порога {threshold:.3f}.")
            else:
                candidates_df = pd.DataFrame(candidate_rows).sort_values(
                    by=["count", "avg_abs_change"], ascending=[False, False]
                )
                display(candidates_df.reset_index(drop=True))

                suggested_years = sorted({row["year"] for row in candidate_rows if row["year"] not in current_shocks})

                print(f"Текущие dummy (shock_years): {current_shocks}")
                print(f"Порог для выделения шоков (|Δ|): {threshold:.3f}")
                if suggested_years:
                    print(f"Рекомендованные годы к добавлению: {suggested_years}")
                else:
                    print("Рекомендация: текущий набор dummy покрывает основные шоки.")



### 9.3 🧭 Отчет по аномалиям макро-рядов

In [ ]:
try:
    with get_data_mart(ROOT, COMPANY) as mart:
        anomalies_report = mart.get_macro_anomalies()
        anomalies_metrics = mart.get_macro_anomaly_metrics()
except Exception as exc:
    print(f"⚠️ Не удалось загрузить отчет по аномалиям: {exc}")
    anomalies_report = pd.DataFrame()
    anomalies_metrics = pd.DataFrame()

if anomalies_report is not None and not anomalies_report.empty:
    display(Markdown("#### macro_anomalies"))
    display(anomalies_report)
else:
    print("ℹ️ Отчет macro_anomalies отсутствует в витрине.")

if anomalies_metrics is not None and not anomalies_metrics.empty:
    display(Markdown("#### macro_anomaly_metrics"))
    display(anomalies_metrics)
else:
    print("ℹ️ Сводные метрики macro_anomaly_metrics отсутствуют или еще не сформированы.")



### 9.4 Проверка трансформаций (dln) в макро-данных

In [ ]:
display(Markdown("### Проверка поддержки dln трансформаций"))

try:
    from engine.model.transforms import apply_transform
    
    # Проверяем что макро-данные могут быть трансформированы
    transform_test_results = []
    
    ecm_cfg = yaml.safe_load(ecm_config_path.read_text(encoding='utf-8')) if ecm_config_path.exists() else {}
    short_run_transform = ecm_cfg.get('run', {}).get('short_run_transform', 'none')
    
    print(f"Short-run transform из конфига: {short_run_transform}")
    
    for factor, data in list(macro_data.items())[:3]:  # Тестируем первые 3
        if data and len(data) >= 2:
            try:
                # Пробуем dln трансформацию
                dln_data = apply_transform(data, 'dln')
                dln_values = [v for v in dln_data.values() if pd.notna(v)]
                
                if dln_values:
                    transform_test_results.append({
                        'factor': factor,
                        'status': '✅',
                        'dln_years': len(dln_values),
                        'dln_mean': float(np.mean(dln_values)),
                        'dln_std': float(np.std(dln_values)),
                        'transform_supported': True
                    })
                    print(f"✅ {factor}: dln трансформация работает ({len(dln_values)} значений)")
                else:
                    transform_test_results.append({
                        'factor': factor,
                        'status': '⚠️',
                        'note': 'Недостаточно данных для dln'
                    })
            except Exception as e:
                transform_test_results.append({
                    'factor': factor,
                    'status': '❌',
                    'error': str(e)[:50]
                })
                print(f"❌ {factor}: Ошибка dln трансформации - {e}")
    
    if transform_test_results:
        transform_df = pd.DataFrame(transform_test_results)
        display(transform_df)
        
except Exception as e:
    print(f"⚠️ Ошибка проверки трансформаций: {e}")
    print("(Это нормально, если transforms модуль не используется в VECM)")



In [ ]:
# Анализ методов моделирования с учетом коинтеграции
if ecm_diag_df.empty:
    print("⚠️ Нет данных о методах моделирования в витрине.")
else:
    fit_analysis_df = ecm_diag_df.copy()
    fit_analysis_df = fit_analysis_df.rename(columns={
        'factor_name': 'factor',
        'block_name': 'block',
        'note': 'note'
    })

    def _method_category(method: str) -> str:
        if method in ['VECM', 'VAR']:
            return '🔷 Мультивариантный'
        if method in ['ARIMA011', 'RW_DRIFT']:
            return '🔶 Univariate ECM'
        if method in ['FALLBACK']:
            return '⚠️ Fallback'
        return '❓ Прочий'

    fit_analysis_df['category'] = fit_analysis_df['method'].apply(_method_category)
    display(fit_analysis_df[['factor', 'method', 'category', 'block', 'p', 'rank', 'span_start', 'span_end', 'lb_pvalue', 'cv_smape', 'note']])

    display(Markdown("\n#### 📊 Статистика методов:"))
    method_counts = fit_analysis_df['method'].value_counts()
    for method, count in method_counts.items():
        pct = count / len(fit_analysis_df) * 100
        print(f"  {method}: {count} фактор(ов) ({pct:.1f}%)")

    univariate_factors = fit_analysis_df[fit_analysis_df['method'].isin(['ARIMA011', 'RW_DRIFT', 'FALLBACK'])]
    if not univariate_factors.empty:
        display(Markdown("\n#### ⚠️ Факторы с univariate ECM:"))
        print("Возможные причины:")
        print("  - Недостаточно истории для VECM")
        print("  - VECM нестабилен (нестабильный прогноз)")
        print("  - Проблемы с коинтеграцией")
        print(f"\nФакторы ({len(univariate_factors)}):")
        for _, row in univariate_factors.iterrows():
            print(f"  - {row['factor']}: {row['method']} ({row.get('note', 'no note')})")
    else:
        print("\n✅ Все факторы используют мультивариантные методы (VECM/VAR)")

    vecm_factors = fit_analysis_df[fit_analysis_df['method'] == 'VECM']
    if not vecm_factors.empty:
        display(Markdown("\n#### 🎯 VECM параметры:"))
        p_values = vecm_factors['p'].dropna().astype(float) if not vecm_factors['p'].dropna().empty else []
        rank_values = vecm_factors['rank'].dropna().astype(float) if not vecm_factors['rank'].dropna().empty else []
        print(f"Факторов с VECM: {len(vecm_factors)}")
        if len(p_values) > 0:
            print(f"Средний lag (p): {np.mean(p_values):.1f}")
        if len(rank_values) > 0:
            print(f"Средний rank: {np.mean(rank_values):.1f}")


## 🔟 Итоговая сводка <a id="section-10"></a>

In [ ]:
total_factors = len(factors)
history_loaded = sum(1 for data in macro_history.values() if data)
forecast_ready = summary_df['forecast_years'].gt(0).sum() if 'summary_df' in locals() and not summary_df.empty else 0

print(f"\n📊 Статистика тестирования:")
print(f"  Всего факторов: {total_factors}")
print(f"  История загружена: {history_loaded}/{total_factors}")
print(f"  Прогнозы с горизонтом: {forecast_ready}/{total_factors}")

if macro_history:
    spans = []
    for factor, data in macro_history.items():
        if data:
            years = sorted(y for y in data.keys() if pd.notna(data[y]))
            if years:
                spans.append(max(years) - min(years) + 1)
    if spans:
        print(f"  Средний span исторических данных: {np.mean(spans):.1f} лет")

all_tests_passed = (history_loaded == total_factors) and (forecast_ready == total_factors)
if all_tests_passed:
    print("\n✅ Модуль макропрогнозирования готов: история и прогнозы присутствуют для всех факторов.")
else:
    print("\n⚠️ Требуется внимание:")
    if history_loaded < total_factors:
        print(f"  ❌ Не удалось загрузить историю для {total_factors - history_loaded} фактор(ов)")
    if forecast_ready < total_factors:
        print(f"  ❌ Недостаточно прогнозного горизонта для {total_factors - forecast_ready} фактор(ов)")

if 'summary_df' in locals() and not summary_df.empty:
    display(Markdown("\n#### Детальная статистика по факторам:"))
    display(summary_df)

# Визуализация прогнозов (первые 4 фактора)
if macro_forecasts:
    display(Markdown("\n#### 📈 Графики прогнозов (ln):"))
    try:
        factors_to_plot = [row['factor'] for _, row in summary_df.iterrows() if row['forecast_years'] > 0][:4]
        if factors_to_plot:
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            axes = axes.flatten()
            for idx, factor in enumerate(factors_to_plot):
                series = pd.Series(macro_forecasts.get(factor, {})).sort_index()
                diag_row = ecm_diag_df[ecm_diag_df['factor_name'] == factor].tail(1)
                span_end = int(diag_row['span_end'].iloc[0]) if not diag_row.empty and pd.notna(diag_row['span_end'].iloc[0]) else None

                history_mask = series.index <= span_end if span_end is not None else series.index <= series.index.max()
                history_series = series[history_mask]
                forecast_series = series[~history_mask] if span_end is not None else pd.Series(dtype=float)

                ax = axes[idx]
                if not history_series.empty:
                    ax.plot(history_series.index, history_series.values, 'o-', label='История (ln)', linewidth=2, markersize=4)
                if not forecast_series.empty:
                    ax.plot(forecast_series.index, forecast_series.values, 's--', label='Прогноз (ln)', linewidth=2, markersize=4, color='green')
                ax.set_xlabel('Год')
                ax.set_ylabel('ln(значение)')
                ax.set_title(factor)
                ax.legend()
                ax.grid(True, alpha=0.3)

            for idx in range(len(factors_to_plot), 4):
                axes[idx].set_visible(False)
            plt.tight_layout()
            plt.show()
        else:
            print("⚠️ Нет факторов с прогнозным горизонтом для визуализации")
    except Exception as e:
        print(f"⚠️ Ошибка построения графиков: {e}")

# Визуализация Actual vs Fitted
if isinstance(ecm_avf_df, pd.DataFrame) and not ecm_avf_df.empty:
    display(Markdown("\n#### 📊 Actual vs Fitted (валидация модели):"))
    try:
        factors_to_plot = ecm_avf_df['factor_name'].unique()[:4]
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        for idx, factor in enumerate(factors_to_plot):
            avf_rows = ecm_avf_df[ecm_avf_df['factor_name'] == factor]
            if avf_rows.empty:
                continue
            ax = axes[idx]
            ax.plot(avf_rows['year'], avf_rows['actual_ln'], 'o-', label='Actual (ln)', linewidth=2, markersize=4)
            ax.plot(avf_rows['year'], avf_rows['fitted_ln'], 's--', label='Fitted (ln)', linewidth=2, markersize=4, color='orange')
            ax.set_xlabel('Год')
            ax.set_ylabel('ln(значение)')
            ax.set_title(factor)
            ax.legend()
            ax.grid(True, alpha=0.3)
        for idx in range(len(factors_to_plot), 4):
            axes[idx].set_visible(False)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"⚠️ Ошибка построения графиков Actual vs Fitted: {e}")


## 1️⃣1️⃣ Итоговый отчет <a id="section-11"></a>


In [ ]:
display(Markdown("### 📊 Итоговый отчет: Методы моделирования и параметры"))

if 'summary_df' in locals() and not summary_df.empty:
    display(Markdown("#### 📈 Статистика методов моделирования:"))
    method_counts = summary_df['method'].value_counts()
    total_factors = len(summary_df)
    print(f"Всего факторов: {total_factors}")
    print("\nИспользованные методы:")
    for method, count in method_counts.items():
        pct = count / total_factors * 100
        print(f"  - {method}: {count} фактор(ов) ({pct:.1f}%)")
    display(Markdown("\n#### 🔍 Детали по каждому фактору:"))
    display(summary_df)
else:
    print("⚠️ Не удалось загрузить summary по прогнозам (витрина пустая или не заполнена).")

if 'fit_analysis_df' in locals() and not fit_analysis_df.empty:
    display(Markdown("\n#### ⚙️ Параметры из ECM diagnostics:"))
    display(fit_analysis_df[['factor', 'method', 'block', 'p', 'rank', 'span_start', 'span_end', 'lb_pvalue', 'cv_smape', 'note']])

if isinstance(ecm_equations_df, pd.DataFrame) and not ecm_equations_df.empty:
    display(Markdown("\n#### 📐 Уравнения коинтеграции (витрина данных):"))
    display(ecm_equations_df)

display(Markdown("\n---"))
display(Markdown("### ✅ Тестирование модуля макропрогнозирования завершено!"))

